In [ ]:
import pandas as pd
import numpy as np
import os
import ast
import seaborn as sns
import matplotlib.pyplot as plt
import json
import math

### Some information : 

For each athlete, we get all of its activities. For each activity, if we have access to the splits we use them because it's way more precise. Therefore we will use data that we can get both from the activities and the splits for the analysis.
Data we will use :
1. In df_final : 

- sport_type (Run, Ride, Swim, Workout...)
- start_date
- athlete_id
- activity_id
- distance
- moving_time
- average_heartrate
- average_speed_km_h

2. In df_athletes_stats : (I am manually adding the data for the athletes I don't have their authorization) 

- athlete_id (or Name Firstname to be decided....)
- Total_distance_run
- Total_distance_ride
- Total_distance_swim
- FC Max (?, or only for the athletes in athletes.json ?)
- 5km PB --> + VDOT equivalent
- 10km PB --> + VDOT equivalent
- 21km PB --> + VDOT equivalent
- 42km PB --> + VDOT equivalent
- VDOT Max (corresponding to their best perf, to "evaluate" their level and compare it to other athletes)
- Nb_activities_run
- Nb_activities_ride
- Nb_activities_swim
- Nb_activities_workout
- Nb_activities

3. df_final : (stats by splits for each activity)
- distance
- moving_time
- sport_type
- activity_id
- start_date
- average_heartrate
- athlete_id
- average_speed_km_h

4. df_best_efforts
- distance_activity
- moving_time_activity
- sport_type
- activity_id
- athlete_id
- id
- best_effort_name
- moving_time
- start_date
- distance_best_effort
- pr_rank

In [ ]:
folder = "../data/raw"
df = pd.concat([pd.read_csv(os.path.join(folder, f)) for f in os.listdir(folder) if f.endswith(".csv")], ignore_index=True) # Concat toutes les data dans un seul df
pd.set_option("display.max_colwidth", 100)

In [ ]:
df.head(3)

## Cleaning

In [ ]:
# Keep only useful columns
keep_cols = ['athlete', 'distance', 'moving_time', 'total_elevation_gain',
       'sport_type', 'id', 'start_date', 'average_speed', 'max_speed',
       'average_watts', 'average_heartrate', 'max_heartrate', 'splits_metric',
       'best_efforts', 'athlete_id', 'max_watts', 'weighted_average_watts']
df = df[keep_cols]
df.head(3)

In [ ]:
# Creating the athlete_id column from the athlete column
df['athlete'] = df['athlete'].apply(ast.literal_eval)
df['athlete_id'] = df['athlete'].apply(lambda x: x['id'] if isinstance(x, dict) else None)
df = df.drop(columns=['athlete'])
df = df.rename(columns={'id': 'activity_id'})

# Speed formatting
df['moving_time'] = df['moving_time'] / 60  # minutes
df['average_speed_km_h'] = df['average_speed'] * 3.6
df['max_speed_km_h_activity'] = df['max_speed'] * 3.6
df = df.drop(columns=["average_speed", 'max_speed'])

df['start_date'] = pd.to_datetime(df['start_date'])

# Renaming some columns to avoid conflicts in splits
df = df.rename(columns={
    'distance': 'distance_activity',
    'moving_time': 'moving_time_activity',
    'average_speed_km_h': 'average_speed_km_h_activity',
    'average_heartrate': 'average_heartrate_activity',
    'total_elevation_gain': 'elevation_gain_activity',
    'max_heartrate': 'max_heartrate_activity',
    'average_watts': 'average_watts_activity',
    'max_watts': 'max_watts_activity',
    'weighted_average_watts': 'weighted_average_watts_activity'
})

# Sort for cumulative distance computation
df = df.sort_values(by=['athlete_id', 'start_date'])

# Compute cumulative distance by sport
def cumulative_distance(df, sport):
    mask = df['sport_type'] == sport
    return (
        df.groupby('athlete_id')['distance_activity']
        .transform(lambda x: x.where(mask).cumsum())
    )

df['cumulative_distance_run'] = cumulative_distance(df, 'Run')
df['cumulative_distance_ride'] = cumulative_distance(df, 'Ride')
df['cumulative_distance_swim'] = cumulative_distance(df, 'Swim')

# splits_metric
df['splits_metric'] = df['splits_metric'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_splits_metric = df.explode("splits_metric").reset_index(drop=True)
df_splits_metric = pd.concat([df_splits_metric.drop(columns=["splits_metric"]), df_splits_metric["splits_metric"].apply(pd.Series)], axis=1)
df = df.drop(columns=["splits_metric"])

# best_efforts
df['best_efforts'] = df['best_efforts'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_efforts = df[df['best_efforts'].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy() # Delete rows without best_efforts
df_best_efforts = df_efforts.explode("best_efforts").reset_index(drop=True) # Explode and keep useful columns
df_best_efforts = pd.concat(
    [df_best_efforts.drop(columns=["best_efforts", 'start_date']), df_best_efforts["best_efforts"].apply(pd.Series)],
    axis=1
)
# Rename/add useful columns
df_best_efforts['elapsed_time_best_effort_min'] = df_best_efforts['elapsed_time'] / 60
df_best_efforts = df_best_efforts.rename(columns={
    'name': 'best_effort_name',
    'elapsed_time': 'elapsed_time_best_effort',
    'distance': 'distance_best_effort'
})

# df_splits_metric
df_splits_metric['start_date'] = pd.to_datetime(df_splits_metric['start_date'])
df_splits_metric['average_speed_km_h'] = df_splits_metric['average_speed'] * 3.6
df_splits_metric = df_splits_metric.rename(columns={
    'distance': 'distance_split',
    'moving_time': 'moving_time_split',
    'average_speed': 'average_speed_split',
    'average_speed_km_h': 'average_speed_km_h_split',
    'average_heartrate': 'average_heartrate_split'
})
df_splits_metric = df_splits_metric.drop(columns=[
    'elapsed_time', 'elevation_difference', 0, 'average_grade_adjusted_speed',
    'cumulative_distance_swim', 'cumulative_distance_ride', 'cumulative_distance_run',
    'pace_zone', 'split'
], errors='ignore')

# delete 'best_efforts' and 'splits_metric' in other dataframes (more convenient)
for col in ['splits_metric', 'best_efforts']:
    for d in [df, df_splits_metric, df_best_efforts]:
        if col in d.columns:
            d.drop(columns=[col], inplace=True)

df_best_efforts = df_best_efforts.dropna(axis=0, subset = ['pr_rank'])  # Delete rows without pr_rank

keep_cols_best_efforts = ['distance_activity', 'moving_time_activity', 'sport_type',
       'activity_id', 'athlete_id', 'id', 'best_effort_name', 'moving_time',
       'start_date', 'distance_best_effort', 'pr_rank']
df_best_efforts = df_best_efforts[keep_cols_best_efforts]
df_best_efforts['start_date'] = pd.to_datetime(df_best_efforts['start_date'])
df_best_efforts['moving_time'] = df_best_efforts['moving_time'] / 60  # minutes
df_best_efforts = df_best_efforts.drop(            # Only keep records on 5, 10, 21 and 42km
    df_best_efforts[
        (df_best_efforts['best_effort_name'] == '2 mile') |
        (df_best_efforts['best_effort_name'] == '1/2 mile') |
        (df_best_efforts['best_effort_name'] == '10 mile') |
        (df_best_efforts['best_effort_name'] == '1K') |
        (df_best_efforts['best_effort_name'] == '400m') |
        (df_best_efforts['best_effort_name'] == '15K') |
        (df_best_efforts['best_effort_name'] == '1 mile')
    ].index
)

In [ ]:
# Step 1 : no splits activities
df_activity_no_splits = df_splits_metric[df_splits_metric['distance_split'].isnull()].copy()

# Step 2 : splits activities
df_activity_only_splits = df_splits_metric[df_splits_metric['distance_split'].notnull()].copy()

# Step 3 : We get rid of all columns related to splits in df_activity_no_splits and the same for columns related to activities in df_activity_only_splits
df_activity_only_splits = df_activity_only_splits.drop(columns=[col for col in df_activity_only_splits.columns if 'activity' in col and col != 'activity_id'], errors='ignore') # We keep 'activity_id' for the fusion
df_activity_no_splits = df_activity_no_splits.drop(columns=[col for col in df_activity_no_splits.columns if 'split' in col], errors='ignore')

# Step 4 : We rename columns to have the same in both dataframes
df_activity_only_splits = df_activity_only_splits.rename(columns={
    'distance_split': 'distance',
    'moving_time_split': 'moving_time',
    'average_speed_km_h_split': 'average_speed_km_h',
    'average_heartrate_split': 'average_heartrate'
})
df_activity_only_splits = df_activity_only_splits.drop(columns=['average_speed_split'])

df_activity_no_splits = df_activity_no_splits.rename(columns={
    'distance_activity': 'distance',
    'moving_time_activity': 'moving_time',
    'average_speed_km_h_activity': 'average_speed_km_h',
    'average_heartrate_activity': 'average_heartrate'
})
df_activity_no_splits = df_activity_no_splits.drop(columns=[col for col in df_activity_no_splits.columns if col not in df_activity_only_splits.columns], errors='ignore') # We keep 'activity_id' for the fusion

# Step 5 : We merge the two dataFrames to have our final df
df_final = pd.concat([df_activity_no_splits, df_activity_only_splits], ignore_index=True)
df_final.dropna(axis=0, subset=['average_heartrate'], inplace=True) # Get rid of rows without average_heartrate (important for our analysis)
df_final.shape

In [ ]:
df_final.head(3)

## Features

In [ ]:
# We create a DataFrame to store athletes stats
df_athletes_stats = pd.DataFrame(columns=['athlete_id', 'Total_distance_run', 'Total_distance_ride', 'Total_distance_swim', 'FC Max', '5km PB', '10km PB', '21km PB',
                                        '42km PB', 'Nb_activities', 'Nb_activities_run', 'Nb_activities_ride', 'Nb_activities_swim'])
df_athletes_stats['Nb_activities_run'] = df_final[df_final['sport_type']=='Run'].groupby('athlete_id')['activity_id'].nunique() # We count the number of activities per athlete for each sport
df_athletes_stats['Nb_activities_ride'] = df_final[df_final['sport_type']=='Ride'].groupby('athlete_id')['activity_id'].nunique()
df_athletes_stats['Nb_activities_swim'] = df_final[df_final['sport_type']=='Swim'].groupby('athlete_id')['activity_id'].nunique()
df_athletes_stats['Nb_activities_workout'] = df_final[df_final['sport_type']=='Workout'].groupby('athlete_id')['activity_id'].nunique()

df_athletes_stats['athlete_id'] = df_final['athlete_id'].unique() # We add athlete IDs

# We look at the total distances covered by each athlete for each sport
total_distance_run = df_final[df_final['sport_type']=='Run'].groupby('athlete_id')['distance'].sum()
total_distance_ride = df_final[df_final['sport_type']=='Ride'].groupby('athlete_id')['distance'].sum()
total_distance_swim = df_final[df_final['sport_type']=='Swim'].groupby('athlete_id')['distance'].sum()
df_athletes_stats['Total_distance_run'] = df_athletes_stats['athlete_id'].map(total_distance_run)
df_athletes_stats['Total_distance_ride'] = df_athletes_stats['athlete_id'].map(total_distance_ride)
df_athletes_stats['Total_distance_swim'] = df_athletes_stats['athlete_id'].map(total_distance_swim)

# We look at the max HR of each athlete (we take the average of the 10 highest max HR, without counting those >210 bpm)
fc_valides = df[df["max_heartrate_activity"] < 210].groupby('athlete_id')["max_heartrate_activity"].apply(lambda x: x.nlargest(10).mean())
df_athletes_stats['FC Max'] = df_athletes_stats['athlete_id'].map(fc_valides)

def extract_best_effort_time(df_best_efforts, distance_km):
    return (
        df_best_efforts[df_best_efforts['distance_best_effort'].between((distance_km - 0.2)*1000, (distance_km + 0.2)*1000)]
        .groupby('athlete_id')['moving_time']
        .min()
        .round(2)
    )

df_athletes_stats['5km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 5))
df_athletes_stats['10km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 10))
df_athletes_stats['21km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 21.1))
df_athletes_stats['42km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 42.2))

In [ ]:
# Manually added stats
manual_stats = pd.read_csv("manual_stats.csv")

# Reorganize columns according to the columns present at load time
common_columns = [col for col in df_athletes_stats.columns if col in manual_stats.columns]
manual_stats = manual_stats[common_columns]

# We concatenate the two DataFrames (pandas will fill missing columns with NaN)
df_athletes_stats = pd.concat([df_athletes_stats, manual_stats], ignore_index=True)

df_athletes_stats = df_athletes_stats.drop_duplicates(subset='athlete_id', keep='last') # Getting rid of duplicates (just in case)

# Then preprocessing of the whole dataframe
df_athletes_stats['Nb_activities_run'] = df_athletes_stats['Nb_activities_run'].fillna(0)
df_athletes_stats['Nb_activities_ride'] = df_athletes_stats['Nb_activities_ride'].fillna(0)
df_athletes_stats['Nb_activities_swim'] = df_athletes_stats['Nb_activities_swim'].fillna(0)
df_athletes_stats['Nb_activities_workout'] = df_athletes_stats['Nb_activities_workout'].fillna(0)
df_athletes_stats['Total_distance_run'] = df_athletes_stats['Total_distance_run'].fillna(0)
df_athletes_stats['Total_distance_ride'] = df_athletes_stats['Total_distance_ride'].fillna(0)
df_athletes_stats['Total_distance_swim'] = df_athletes_stats['Total_distance_swim'].fillna(0)
df_athletes_stats['Nb_activities'] = df_athletes_stats['Nb_activities_run'] + df_athletes_stats['Nb_activities_ride'] + df_athletes_stats['Nb_activities_swim'] + df_athletes_stats['Nb_activities_workout']


# Transforming each PB into its equivalent VDOT score (approximation) to better compare performances (Daniels, J. (2013). Daniels’ Running Formula (3rd ed.). Human Kinetics)
def calculate_vdot(distance_m, time_sec):
    time_min = time_sec / 60
    if time_min == 0:
        return None

    velocity = distance_m / time_min  # m/min
    vo2 = -4.6 + 0.182258 * velocity + 0.000104 * velocity**2

    percent_max = 0.8 + 0.1894393 * math.exp(-0.012778 * time_min) + \
                  0.2989558 * math.exp(-0.1932605 * time_min)

    vdot = vo2 / percent_max
    return round(vdot, 2)

for dist, colname in [(5000, '5km PB'), (10000, '10km PB'), (21100, '21km PB'), (42200, '42km PB')]:
    vdot_col = f'VDOT_{colname.split()[0]}'
    df_athletes_stats[vdot_col] = df_athletes_stats.apply(
        lambda row: calculate_vdot(dist, row[colname]*60) if pd.notna(row[colname]) else None, axis=1)

# And then we create the VDOT max variable to "rate" the athletes and standardize performances
df_athletes_stats['VDOT_max'] = df_athletes_stats[['VDOT_5km', 'VDOT_10km', 'VDOT_21km', 'VDOT_42km']].max(axis=1)

In [ ]:
    # We create a 'cv_speed' variable which corresponds to the Coefficient of Variation of speed for each activity (to better differentiate interval training sessions for example: if cv is high, it means
    # that the athlete varied his speed a lot, so he did interval training, if cv is low, he did a steady pace run) 
    # We also create a zone variable which corresponds to the heart rate zone (Z1, Z2, Z3, Z4) for each activity (I used the model from Leif Inge Tjelta for the zones, except that I combined Z4 and Z5)
    # All of these features will be useful for clustering later
def process_splits(splits_df, fc_max_dict):
    # Takes as input:
    # - splits_df: a DataFrame containing the splits of activities (with columns 'activity_id', 'athlete_id', 'average_heartrate', 'average_speed_km_h')
    # - fc_max_dict: a dictionary {athlete_id: fc_max}
    # Returns:
    # - A DataFrame with for each activity:
    #     - cv_speed: coefficient of variation of speed
    #     - pct_Z1 to pct_Z4: percentage of time spent in each HR zone
     splits_df = splits_df.copy()

    # We define the HR zone thresholds as a percentage of max HR
     def get_hr_zone(hr, fc_max):
        if pd.isna(hr) or pd.isna(fc_max):
            return None
        ratio = hr / fc_max
        if ratio < 0.82:
            return "Z1"
        elif ratio < 0.92:
            return "Z2"
        elif ratio < 0.97:
            return "Z3"
        else:
            return "Z4"

    # Apply the heart rate zone to each row
     splits_df["fc_max"] = splits_df["athlete_id"].map(fc_max_dict)
     splits_df["hr_zone"] = splits_df.apply(lambda row: get_hr_zone(row["average_heartrate"], row["fc_max"]), axis=1)

     results = []
     for activity_id, group in splits_df.groupby("activity_id"):
        # Coefficient of variation of speed
        cv_speed = group["average_speed_km_h"].std() / group["average_speed_km_h"].mean()

        # Percentage by heart rate zone
        zone_pct = group["hr_zone"].value_counts(normalize=True).reindex(["Z1", "Z2", "Z3", "Z4"], fill_value=0)
        zone_pct.index = [f"pct_{z}" for z in zone_pct.index]

        res = {"activity_id": activity_id, "cv_speed": cv_speed}
        res.update(zone_pct.to_dict())
        results.append(res)

     return pd.DataFrame(results)


# Create a dictionary to store the max HR of each athlete
fc_max_dict = df_athletes_stats.set_index("athlete_id")["FC Max"].to_dict()
# Applying the function on the splits/activities df
df_processed_splits = process_splits(df_final, fc_max_dict)
# And then we merge
df_final = df_final.merge(df_processed_splits, on="activity_id", how="left")
df_final['cv_speed'].fillna(0, inplace=True) # We suppose that whole activities (without splits) have a cv of 0, so at a constant pace


In [ ]:
df_final.tail(5)
# df_final.shape

In [ ]:
df_final_by_activity = df_final[['athlete_id', 'activity_id', 'start_date', 'cv_speed', 'pct_Z1', 'pct_Z2', 'pct_Z3', 'pct_Z4', 'sport_type']].copy() # On garde que les colonnes utiles pour regrouper par activités
df_final_by_activity['distance'] = df_final.groupby('activity_id')['distance'].transform('sum')
df_final_by_activity['moving_time'] = df_final.groupby('activity_id')['moving_time'].transform('sum')
df_final_by_activity = df_final_by_activity.drop_duplicates(subset='activity_id') # On enlève les doublons (car plusieurs splits pour une même activité)

# On ajoute les variables "Intensité" et "Charge d'entraînement" (à ajuster)
# Autre option : training_load = intensity * moving_time * (1 + average_speed_km_h / 20)
# un truc du genre pour prendre en compte la vitesse
# OU internal load avec FC, cv_speed... et external load avec distance/moving_time, vitesse...
df_final_by_activity['intensity'] = (df_final_by_activity["pct_Z1"] * 1 + df_final_by_activity["pct_Z2"] * 1.5 + df_final_by_activity["pct_Z3"] * 2.25 + df_final_by_activity["pct_Z4"] * 3) #+ df_final_by_activity['cv_speed'] * 2
df_final_by_activity['training_load'] = df_final_by_activity['intensity'] * df_final_by_activity['distance']

df_final_by_activity.tail()

In [ ]:
df_athletes_stats.head()
# df_athletes_stats.shape

In [ ]:
df_run = df_final_by_activity[df_final_by_activity['sport_type'] == 'Run'] # On divise on 4 datasets, pour chaque type d'activité
df_bike = df_final_by_activity[df_final_by_activity['sport_type'] == 'Ride']
df_swim = df_final_by_activity[df_final_by_activity['sport_type'] == 'Swim']
df_workout = df_final_by_activity[df_final_by_activity['sport_type'] == 'Workout']

In [ ]:
df_run = df_run.drop(columns = ['sport_type'])
df_bike = df_bike.drop(columns = ['sport_type'])
df_swim = df_swim.drop(columns = ['sport_type'])
df_workout = df_workout.drop(columns = ['sport_type'])

In [ ]:
total_distance = df_final_by_activity.groupby('athlete_id')['distance'].sum()
total_distance = pd.DataFrame(total_distance).reset_index()
total_distance = total_distance.rename(columns = {0 : 'total_distance'})
total_distance

In [ ]:
unique_counts = df_final_by_activity.nunique().sort_values(ascending=False) # Compte le nombre de valeurs uniques dans chaque colonne
df_unique = pd.DataFrame({'Colonne': unique_counts.index, 'Valeurs Uniques': unique_counts.values})
print(df_unique)

In [ ]:
missing_values_count = df_run.isnull().sum() # Compte le nombre de valeurs nulles pour chaque variable
missing_values_count.sort_values(ascending=False)

In [ ]:
df_values = df_best_efforts[df_best_efforts['athlete_id'] == 118945026]["best_effort_name"].value_counts().reset_index() # Compte le nombre d'occcurences différentes pour chaque valeur d'une certaine variable
df_values.columns = ["Valeur", "Occurrences"]
df_values.sort_values(by='Valeur', ascending = False).head(10)

## #1 Running activities analysis ###

### Univariate analysis (activities)

In [ ]:
df_run.dtypes

In [ ]:
df_run.shape

In [ ]:
unique_counts_run = df_run.nunique().sort_values(ascending=False)
df_unique_run = pd.DataFrame({'Colonne': unique_counts_run.index, 'Valeurs Uniques': unique_counts_run.values})
print(df_unique_run)

In [ ]:
numerical_var = df_run[['average_heartrate', 'distance', 'moving_time', 'average_speed_km_h']]

fig = plt.figure(figsize=(18, 16))

for index, col in enumerate(numerical_var.columns, 1):
    plt.subplot(3, 2, index)
    sns.histplot(df_run[col].dropna(), kde=False, bins=50)
    plt.title(col)

fig.tight_layout(pad=1.0)
plt.show()

In [ ]:
# categorical_var = df_run.select_dtypes(exclude=['number'])
# categorical_var = categorical_var.drop(columns=['start_date'])
# fig = plt.figure(figsize=(18, 16))

# for index, col in enumerate(categorical_var.columns, 1):
#     plt.subplot(6, 4, index)
#     sns.countplot(df_run[col])
#     plt.xticks(rotation = 90)
#     plt.title(col)

# fig.tight_layout(pad=1.0)
# plt.show()

### Univariate analysis (athletes_stats)

In [ ]:
numerical_var2 = df_athletes_stats[['Total_distance_run', 'Total_distance_ride', 'Total_distance_swim', 'VDOT_5km', 'VDOT_10km', 'VDOT_21km', 'VDOT_42km', 'VDOT_max',
                                     'Nb_activities', 'Nb_activities_run', 'Nb_activities_ride', 'Nb_activities_swim', 'Nb_activities_workout']]

fig = plt.figure(figsize=(18, 16))

for index, col in enumerate(numerical_var2.columns, 1):
    plt.subplot(4, 4, index)
    sns.histplot(df_athletes_stats[col].dropna(), kde=False, bins=50)
    plt.title(col)

fig.tight_layout(pad=1.0)
plt.show()

### Bivariate analysis

In [ ]:
plt.figure(figsize=(10,6))
correlation = numerical_var2.corr()
sns.heatmap(correlation, linewidths=0.5, cmap='Blues', annot=True)

### Clustering

In [ ]:
# Ici on va essayer de faire du clustering sur toutes les séances des athlètes : différencier les séances d'EF, seuil, fractio etc...
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [ ]:
X = df_run[['activity_id', 'cv_speed', 'pct_Z1', 'pct_Z2', 'pct_Z3', 'pct_Z4', 'intensity', 'training_load', 'distance']]
X = X.drop_duplicates(subset='activity_id') # On enlève les doublons (caar plusieurs splits pour une même activité)
X = X.drop(columns=['activity_id'])

X_scaled = StandardScaler().fit_transform(X) # On normalise les données car les unités sont différentes

In [ ]:
inertias = []
for i in range(1, 11): # Méthode du coude pour trouver k (nb de clusters optimal)
    kmeans = KMeans(n_clusters=i, random_state=0)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
plt.plot(range(1, 11), inertias)
plt.title('Méthode du coude')
plt.xlabel('Nombre de clusters')
plt.ylabel('Inertie')
plt.show()

In [ ]:
silhouette_scores = []
for i in range(2, 11):  # On teste aussi avec la méthode des silhouettes (on recherche le meilleur score)
    kmeans = KMeans(n_clusters=i, random_state=0)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.plot(range(2, 11), silhouette_scores)
plt.title('Silhouette score selon le nombre de clusters')
plt.xlabel('Nombre de clusters')
plt.ylabel('Silhouette score')
plt.show()

### Interprétation des clusters

In [ ]:
# Ici K=5 semble le plus pertinent
kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(X_scaled)

In [ ]:
X['cluster'] = kmeans.labels_
cluster_summary = X.groupby('cluster').mean()
display(cluster_summary)

In [ ]:
df_clusters = df_run[['activity_id', 'athlete_id']].drop_duplicates()
df_clusters['cluster'] = kmeans.labels_

# On groupe par athlète pour voir les proportions de ses activités dans chaque cluster
df_clusters.groupby('athlete_id')['cluster'].value_counts(normalize=True).unstack().fillna(0)


In [ ]:
X['cluster'].value_counts() # Nombre d'activités par cluster

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_plot = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_plot['cluster'] = kmeans.labels_

sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='cluster', palette='tab10')

In [ ]:
X.sort_values(by='training_load').tail()

In [ ]:
df_run['cluster'] = df_clusters['cluster']

In [ ]:
# df_run.tail(10)
df_run.dtypes

In [ ]:
df_athletes_stats.dtypes

In [ ]:
df_run = df_run.sort_values(by=["athlete_id", "start_date"]).copy()

# Temps passé dans chaque zone (en secondes)
df_run["time_Z1"] = df_run["pct_Z1"] * df_run["moving_time"]
df_run["time_Z2"] = df_run["pct_Z2"] * df_run["moving_time"]
df_run["time_Z3"] = df_run["pct_Z3"] * df_run["moving_time"]
df_run["time_Z4"] = df_run["pct_Z4"] * df_run["moving_time"]

# Cumul par athlète
df_run["cumulative_time"] = df_run.groupby("athlete_id")["moving_time"].cumsum()
df_run["cumulative_Z1"] = df_run.groupby("athlete_id")["time_Z1"].cumsum()
df_run["cumulative_Z2"] = df_run.groupby("athlete_id")["time_Z2"].cumsum()
df_run["cumulative_Z3"] = df_run.groupby("athlete_id")["time_Z3"].cumsum()
df_run["cumulative_Z4"] = df_run.groupby("athlete_id")["time_Z4"].cumsum()

# Pourcentage cumulé de temps passé dans chaque zone
df_run["pct_time_Z1"] = (df_run["cumulative_Z1"] / df_run["cumulative_time"] * 100).round(2)
df_run["pct_time_Z2"] = (df_run["cumulative_Z2"] / df_run["cumulative_time"] * 100).round(2)
df_run["pct_time_Z3"] = (df_run["cumulative_Z3"] / df_run["cumulative_time"] * 100).round(2)
df_run["pct_time_Z4"] = (df_run["cumulative_Z4"] / df_run["cumulative_time"] * 100).round(2)


In [ ]:
df_run["start_date"] = pd.to_datetime(df_run["start_date"])
df_run = df_run.sort_values(by=["athlete_id", "start_date"])
df_run = df_run.set_index("start_date")


df_run['cumulative_training_load_2_weeks'] = (
    df_run.groupby('athlete_id')['training_load']
    .rolling('14D', min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df_run['cumulative_training_load_4_weeks'] = (
    df_run.groupby('athlete_id')['training_load']
    .rolling('28D', min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df_run['cumulative_training_load_8_weeks'] = (
    df_run.groupby('athlete_id')['training_load']
    .rolling('56D', min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)
df_run.reset_index(inplace=True)

In [ ]:
def plot_variable_for_athlete(df, athlete_id, variable, title=None, ylabel=None):
    athlete_data = df[df["athlete_id"] == athlete_id]

    plt.figure(figsize=(12, 5))
    plt.plot(athlete_data.index, athlete_data[variable], marker='o', linestyle='-')
    plt.title(title or f"{variable} over time for athlete {athlete_id}")
    plt.xlabel("Date")
    plt.ylabel(ylabel or variable)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_variable_for_athlete(df=df_run,athlete_id="118945026",variable="pct_time_Z1",title="Charge d'entraînement cumulée sur 4 semaines",ylabel="Training Load")

In [ ]:
# Sauvegarde des DataFrames dans des fichiers CSV

df_run.to_csv("Data/processed/df_run.csv", index=False)
df_athletes_stats.to_csv("Data/processed/df_athletes_stats.csv", index=False)